# GAN Variants and Modern Practice

If the original GAN chapter introduces the adversarial game, this chapter should explain why one almost never stops at the vanilla minimax loss in practice. The standard GAN is elegant, but its optimization signal can be fragile, the discriminator can become too sharp or too unstable, and the entire game can fail to cover the true data distribution well.

For teaching purposes, it is better to study **a small number of central GAN variants well** than to survey a large zoo. We therefore focus on three common branches: **WGAN-GP**, **spectral normalization**, and **conditional GANs**. Together, they cover the three main axes of adversarial modeling: better geometry, better critic control, and controlled generation.

This chapter treats the main GAN variants as **theory plus runnable pattern**. For each one, the structure is the same: explain what failure mode motivates the variant, show the code change, train the model on `FashionMNIST`, inspect samples or interpolations, and postpone FID/KID to a shared final comparison so the real-data features are computed only once.

The training budgets below are intentionally more serious than toy defaults. If a GAN variant is only trained for a handful of epochs, the reader mostly learns what undertraining looks like. The point here is to give each variant a real chance to generate something meaningful.


## Shared Setup

We keep the dataset, generator backbone, and most of the optimizer conventions fixed whenever possible. That way, differences in behavior are easier to attribute to the variant itself rather than to unrelated changes in the pipeline.

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchvision import datasets, transforms, utils
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
if device.type == "cuda":
    torch.cuda.manual_seed_all(7)
num_workers = 2 if device.type == "cuda" else 0
project_root = Path.cwd() if (Path.cwd() / "_config.yml").exists() else Path.cwd().parent
DATA_ROOT = project_root / "data"

# DCGAN-style settings are still teachable but produce much better samples.
latent_dim = 128
base_channels = 64
batch_size = 128
lr = 2e-4
epochs = 60

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 2.0 * x - 1.0),
])

train_dataset = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)


## WGAN-GP

The theoretical motivation for `WGAN-GP` is already in the previous discussion: better geometry and smoother gradients. The implementation consequence is that the discriminator becomes a **critic** and the loss acquires a gradient penalty term.

## From the Vanilla GAN to Wasserstein Geometry

The most important conceptual shift in the GAN literature is the move from the Jensen-Shannon geometry of the original game to the **Wasserstein-1 distance**. In dual form,
```{math}
W_1(p_{gt}, p_\theta)
=
\sup_{\|f\|_L \le 1}
\mathbb{E}_{\boldsymbol{x}\sim p_{gt}}[f(\boldsymbol{x})]
-
\mathbb{E}_{\boldsymbol{x}\sim p_\theta}[f(\boldsymbol{x})].
```
The discriminator is replaced by a **critic**, and the key mathematical condition becomes **1-Lipschitz continuity**.

Why is this such a big deal? Because Wasserstein distance can still change smoothly even when the supports of the real and generated distributions barely overlap. That is precisely the regime where the original GAN often struggles most, especially early in training. In that sense, WGAN is not just a variant with a different formula. It is a different view of what signal the generator ought to receive.

A simple mental picture helps. Imagine that the true data are images of shoes and the generator initially produces something like amorphous grayscale blobs. Under the original adversarial objective, the discriminator may confidently separate the two distributions, leaving little useful gradient structure. Under the Wasserstein view, the critic is instead encouraged to estimate **how far probability mass must move** to turn the blobs into plausible shoes. That tends to produce a more graded training signal.

The catch is that the theory requires the critic to be 1-Lipschitz. The original WGAN enforced this with **weight clipping**, which is historically important but often a poor numerical approximation. That leads directly to the most standard practical version.

In [ ]:
class WGANGenerator(nn.Module):
    def __init__(self, latent_dim=128, base_channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, base_channels * 4 * 7 * 7),
            nn.BatchNorm1d(base_channels * 4 * 7 * 7),
            nn.ReLU(True),
            nn.Unflatten(1, (base_channels * 4, 7, 7)),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(True),
            nn.Conv2d(base_channels, 1, kernel_size=3, padding=1),
            nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z)

class WGANCritic(nn.Module):
    def __init__(self, base_channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, base_channels, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(base_channels * 2 * 7 * 7, 1),
        )
    def forward(self, x):
        return self.net(x)


In [ ]:
def gradient_penalty(critic, real_images, fake_images, device):
    batch_n = real_images.size(0)
    alpha = torch.rand(batch_n, 1, 1, 1, device=device)
    interpolated = alpha * real_images + (1 - alpha) * fake_images
    interpolated.requires_grad_(True)

    critic_scores = critic(interpolated)
    grad_outputs = torch.ones_like(critic_scores)
    gradients = torch.autograd.grad(
        outputs=critic_scores,
        inputs=interpolated,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(batch_n, -1)
    return ((gradients.norm(2, dim=1) - 1.0) ** 2).mean()


def wgan_critic_loss(critic, real_images, fake_images, gp_weight=10.0):
    real_score = critic(real_images).mean()
    fake_score = critic(fake_images).mean()
    gp = gradient_penalty(critic, real_images, fake_images, real_images.device)
    loss = fake_score - real_score + gp_weight * gp
    return loss, real_score, fake_score, gp


def wgan_generator_loss(critic, fake_images):
    return -critic(fake_images).mean()


# In a real WGAN-GP loop, the critic is often updated several times per generator step.
# The critic sees detached fake images, while the generator update uses fresh fake samples.


In [ ]:
wgan_G = WGANGenerator(latent_dim=latent_dim, base_channels=base_channels).to(device)
wgan_D = WGANCritic(base_channels=base_channels).to(device)
wgan_g_optimizer = torch.optim.Adam(wgan_G.parameters(), lr=lr, betas=(0.5, 0.999))
wgan_d_optimizer = torch.optim.Adam(wgan_D.parameters(), lr=lr, betas=(0.5, 0.999))
wgan_epochs = 50
critic_steps = 5
wgan_history = {"d_loss": [], "g_loss": []}

for epoch in tqdm(range(wgan_epochs), desc="WGAN-GP epochs"):
    d_running = 0.0
    g_running = 0.0
    for real_images, _ in tqdm(train_loader, desc="WGAN train", leave=False):
        real_images = real_images.to(device)
        batch_n = real_images.size(0)
        for _ in range(critic_steps):
            z = torch.randn(batch_n, latent_dim, device=device)
            fake_images = wgan_G(z)
            wgan_d_optimizer.zero_grad()
            d_loss, _, _, _ = wgan_critic_loss(wgan_D, real_images, fake_images.detach(), gp_weight=10.0)
            d_loss.backward()
            wgan_d_optimizer.step()
        z = torch.randn(batch_n, latent_dim, device=device)
        fake_images = wgan_G(z)
        wgan_g_optimizer.zero_grad()
        g_loss = wgan_generator_loss(wgan_D, fake_images)
        g_loss.backward()
        wgan_g_optimizer.step()
        d_running += d_loss.item()
        g_running += g_loss.item()
    wgan_history["d_loss"].append(d_running / len(train_loader))
    wgan_history["g_loss"].append(g_running / len(train_loader))
    print(
        f"Epoch {epoch + 1:02d} | "
        f"critic loss: {wgan_history['d_loss'][-1]:.4f} | "
        f"generator loss: {wgan_history['g_loss'][-1]:.4f}"
    )


Because `WGAN-GP` is still an unconditional model, we can evaluate it with the same kinds of sample grids and latent interpolations used in the vanilla GAN setting. The main visual difference one hopes for is not “sharper than everything else,” but more stable improvement and less obvious collapse.

In [ ]:
@torch.no_grad()
def show_gan_samples(generator, device, n=16):
    generator.eval()
    z = torch.randn(n, latent_dim, device=device)
    samples = 0.5 * (generator(z) + 1.0)
    image = utils.make_grid(samples.cpu(), nrow=4, pad_value=1.0)
    plt.figure(figsize=(6, 6))
    plt.imshow(image.permute(1, 2, 0), cmap='gray')
    plt.axis('off')
    plt.show()

@torch.no_grad()
def show_gan_interpolation(generator, device, steps=8):
    generator.eval()
    z0 = torch.randn(1, latent_dim, device=device)
    z1 = torch.randn(1, latent_dim, device=device)
    alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1)
    z = (1 - alphas) * z0 + alphas * z1
    samples = 0.5 * (generator(z) + 1.0)
    image = utils.make_grid(samples.cpu(), nrow=steps, pad_value=1.0)
    plt.figure(figsize=(1.7 * steps, 2.5))
    plt.imshow(image.permute(1, 2, 0), cmap='gray')
    plt.axis('off')
    plt.show()

show_gan_samples(wgan_G, device)
show_gan_interpolation(wgan_G, device)


## Spectral Normalization GAN

Spectral normalization leaves the outer game close to the original non-saturating GAN, but changes the function class of the discriminator. That makes it a very clean comparison point with `WGAN-GP`: one variant mostly changes the loss geometry, the other mostly changes critic control.


**Spectral normalization** is a different answer to the same broad question: how do we keep the critic from becoming too wild? Instead of adding a penalty term, we constrain each layer by dividing its weight matrix by an estimate of its largest singular value.

This makes spectral normalization one of the cleanest examples of an **architectural regularization** technique. The loss can stay almost the same, yet the critic class becomes much better behaved. In practice, this is attractive because it is easy to add and often stabilizes training with very little extra machinery.

From a teaching perspective, spectral normalization is valuable because it broadens the students' mental model. GAN stability is not only about choosing the right objective. It is also about controlling the geometry of the discriminator network itself.

In [ ]:
class SNDiscriminator(nn.Module):
    def __init__(self, base_channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(1, base_channels, kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2, inplace=True),
            nn.utils.spectral_norm(
                nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.utils.spectral_norm(
                nn.Linear(base_channels * 2 * 7 * 7, 1)
            ),
        )

    def forward(self, x):
        return self.net(x)


sn_discriminator = SNDiscriminator(base_channels=base_channels).to(device)
sn_optimizer = torch.optim.Adam(sn_discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

In [ ]:
sn_G = WGANGenerator(latent_dim=latent_dim, base_channels=base_channels).to(device)
sn_D = SNDiscriminator(base_channels=base_channels).to(device)
sn_g_optimizer = torch.optim.Adam(sn_G.parameters(), lr=lr, betas=(0.5, 0.999))
sn_d_optimizer = torch.optim.Adam(sn_D.parameters(), lr=lr, betas=(0.5, 0.999))
sn_epochs = 50
sn_history = {"d_loss": [], "g_loss": []}

for epoch in tqdm(range(sn_epochs), desc="SN-GAN epochs"):
    d_running = 0.0
    g_running = 0.0
    for real_images, _ in tqdm(train_loader, desc="SN train", leave=False):
        real_images = real_images.to(device)
        batch_n = real_images.size(0)
        z = torch.randn(batch_n, latent_dim, device=device)
        fake_images = sn_G(z)
        sn_d_optimizer.zero_grad()
        real_logits = sn_D(real_images)
        fake_logits = sn_D(fake_images.detach())
        d_loss = discriminator_loss(real_logits, fake_logits)
        d_loss.backward()
        sn_d_optimizer.step()
        z = torch.randn(batch_n, latent_dim, device=device)
        fake_images = sn_G(z)
        sn_g_optimizer.zero_grad()
        fake_logits = sn_D(fake_images)
        g_loss = generator_loss(fake_logits)
        g_loss.backward()
        sn_g_optimizer.step()
        d_running += d_loss.item()
        g_running += g_loss.item()
    sn_history["d_loss"].append(d_running / len(train_loader))
    sn_history["g_loss"].append(g_running / len(train_loader))
    print(
        f"Epoch {epoch + 1:02d} | "
        f"D loss: {sn_history['d_loss'][-1]:.4f} | "
        f"G loss: {sn_history['g_loss'][-1]:.4f}"
    )


In [ ]:
show_gan_samples(sn_G, device)
show_gan_interpolation(sn_G, device)


## Conditional GANs

The conditional GAN changes the task itself. Instead of matching one unconditional image distribution, the model must now match a **family of class-conditional distributions**. The label controls what kind of object must be present, while the latent noise still controls which specific instance appears.


A third standard branch is the **conditional GAN (cGAN)**. Here the target is no longer an unconditional image distribution, but a conditional one. The generator receives noise together with some condition $\boldsymbol{y}$, and the discriminator judges both realism and condition compatibility:
```{math}
\min_G \max_D
\mathbb{E}_{(\boldsymbol{x},\boldsymbol{y})\sim p_{gt}}[\log D(\boldsymbol{x},\boldsymbol{y})]
+
\mathbb{E}_{\boldsymbol{z}\sim p(\boldsymbol{z}),\boldsymbol{y}\sim p(\boldsymbol{y})}
[\log(1-D(G(\boldsymbol{z},\boldsymbol{y}),\boldsymbol{y}))].
```

This is important because it shows that adversarial learning is not just a tool for sampling from noise. It is a tool for **controlled synthesis**. If $\boldsymbol{y}$ is a class label, the model should produce an image of the requested class. If $\boldsymbol{y}$ is another image, one enters the larger family of image-to-image translation models.

In [ ]:
num_classes = 10


class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim=128, base_channels=64, num_classes=10, label_dim=16):
        super().__init__()
        self.label_embed = nn.Embedding(num_classes, label_dim)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + label_dim, base_channels * 4 * 7 * 7),
            nn.BatchNorm1d(base_channels * 4 * 7 * 7),
            nn.ReLU(True),
            nn.Unflatten(1, (base_channels * 4, 7, 7)),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(True),
            nn.Conv2d(base_channels, 1, kernel_size=3, padding=1),
            nn.Tanh(),
        )

    def forward(self, z, y):
        y_embed = self.label_embed(y)
        return self.net(torch.cat([z, y_embed], dim=1))


class ConditionalDiscriminator(nn.Module):
    def __init__(self, base_channels=64, num_classes=10, label_dim=16):
        super().__init__()
        self.label_embed = nn.Embedding(num_classes, 28 * 28)
        self.net = nn.Sequential(
            nn.Conv2d(2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(base_channels * 2 * 7 * 7, 1),
        )

    def forward(self, x, y):
        y_map = self.label_embed(y).view(-1, 1, 28, 28)
        return self.net(torch.cat([x, y_map], dim=1))


cG = ConditionalGenerator(latent_dim=latent_dim, base_channels=base_channels, num_classes=num_classes).to(device)
cD = ConditionalDiscriminator(base_channels=base_channels, num_classes=num_classes).to(device)

# Typical usage:
# labels = labels.to(device)
# z = torch.randn(labels.size(0), latent_dim, device=device)
# fake_images = cG(z, labels)
# real_logits = cD(real_images, labels)
# fake_logits = cD(fake_images.detach(), labels)


In [ ]:
cG = ConditionalGenerator(latent_dim=latent_dim, base_channels=base_channels, num_classes=num_classes).to(device)
cD = ConditionalDiscriminator(base_channels=base_channels, num_classes=num_classes).to(device)
cgan_g_optimizer = torch.optim.Adam(cG.parameters(), lr=lr, betas=(0.5, 0.999))
cgan_d_optimizer = torch.optim.Adam(cD.parameters(), lr=lr, betas=(0.5, 0.999))
conditional_gan_epochs = 50
conditional_gan_history = {"d_loss": [], "g_loss": []}

for epoch in tqdm(range(conditional_gan_epochs), desc="cGAN epochs"):
    d_running = 0.0
    g_running = 0.0

    for real_images, labels in tqdm(train_loader, desc="cGAN train", leave=False):
        real_images = real_images.to(device)
        labels = labels.to(device)
        batch_n = real_images.size(0)

        z = torch.randn(batch_n, latent_dim, device=device)
        fake_images = cG(z, labels)

        cgan_d_optimizer.zero_grad()
        real_logits = cD(real_images, labels)
        fake_logits = cD(fake_images.detach(), labels)
        d_loss = discriminator_loss(real_logits, fake_logits)
        d_loss.backward()
        cgan_d_optimizer.step()

        z = torch.randn(batch_n, latent_dim, device=device)
        fake_images = cG(z, labels)

        cgan_g_optimizer.zero_grad()
        fake_logits = cD(fake_images, labels)
        g_loss = generator_loss(fake_logits)
        g_loss.backward()
        cgan_g_optimizer.step()

        d_running += d_loss.item()
        g_running += g_loss.item()

    d_epoch = d_running / len(train_loader)
    g_epoch = g_running / len(train_loader)
    conditional_gan_history["d_loss"].append(d_epoch)
    conditional_gan_history["g_loss"].append(g_epoch)

    print(
        f"Epoch {epoch + 1:02d} | cD loss: {d_epoch:.4f} | cG loss: {g_epoch:.4f}"
    )


The conditional discriminator now has a stricter job than the unconditional one. It must reject not only unrealistic images, but also **mismatched** image-label pairs. A generated boot with the label `T-shirt/top` should count as wrong even if it looks locally plausible as an image.

In [ ]:
@torch.no_grad()
def show_cgan_class_samples(generator, class_names, device, n_per_class=6):
    generator.eval()
    rows = []
    for class_id, _ in enumerate(class_names):
        labels = torch.full((n_per_class,), class_id, device=device, dtype=torch.long)
        z = torch.randn(n_per_class, latent_dim, device=device)
        samples = generator(z, labels)
        samples = 0.5 * (samples + 1.0)
        rows.append(samples.cpu())

    image = utils.make_grid(torch.cat(rows, dim=0), nrow=n_per_class, pad_value=1.0)
    plt.figure(figsize=(1.6 * n_per_class, 1.6 * len(class_names)))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()

    for class_id, name in enumerate(class_names):
        print(f"row {class_id}: {name}")


@torch.no_grad()
def interpolate_cgan_with_fixed_class(generator, class_id, class_name, device, steps=8):
    generator.eval()
    z0 = torch.randn(1, latent_dim, device=device)
    z1 = torch.randn(1, latent_dim, device=device)
    alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1)
    z = (1 - alphas) * z0 + alphas * z1
    labels = torch.full((steps,), class_id, device=device, dtype=torch.long)
    samples = 0.5 * (generator(z, labels) + 1.0)
    image = utils.make_grid(samples.cpu(), nrow=steps, pad_value=1.0)
    plt.figure(figsize=(1.7 * steps, 2.5))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()
    print(f"class-conditioned interpolation: {class_name}")


show_cgan_class_samples(cG, train_dataset.classes, device, n_per_class=6)
interpolate_cgan_with_fixed_class(cG, class_id=9, class_name=train_dataset.classes[9], device=device)


Here the expectation is again class-wise coherence with intra-class variety. The `Pullover` row should mostly stay in the pullover family, but different columns should still vary in collar, sleeve width, texture, or silhouette. If every row looks nearly identical internally, then the conditioning may be working while diversity is failing. If rows look mixed across classes, then the label signal is not being used strongly enough.

The interpolation experiment checks the same idea from a different angle. Once the class is fixed, moving through latent space should change style and local structure while preserving category identity. If the interpolation quietly crosses semantic classes, then the generator is not really separating condition information from residual variation.

For this course, these are the GAN variants most worth implementing in detail. `WGAN-GP` teaches better critic geometry, `spectral normalization` teaches critic control, and `cGAN` teaches controlled generation. Other important families such as `CycleGAN` or `StyleGAN` are better treated as follow-on readings once these three levers are understood.


## Shared FID/KID Comparison With Cached Real Features

As in the VAE variants notebook, we centralize the metric computation here. The helper below computes the real-image features once and then reuses them for the fake-image passes of all three variants.

In [ ]:
import copy
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

def prepare_for_inception_metrics(images):
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)
    return images.clamp(0.0, 1.0)

@torch.no_grad()
def build_cached_real_metrics(real_loader, device, num_real=1000, metric_batch_size=64):
    fid = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).set_dtype(torch.float64).to(device)
    kid = KernelInceptionDistance(feature=2048, subsets=10, subset_size=100, normalize=True, reset_real_features=False).to(device)
    seen = 0
    for real_images, _ in tqdm(real_loader, desc='real metric cache', leave=False):
        remaining = num_real - seen
        if remaining <= 0:
            break
        real_images = real_images[: min(metric_batch_size, remaining)].to(device)
        real_images = prepare_for_inception_metrics(0.5 * (real_images + 1.0))
        fid.update(real_images, real=True)
        kid.update(real_images, real=True)
        seen += real_images.size(0)
    return fid, kid

@torch.no_grad()
def evaluate_with_cached_real(sample_fn, base_fid, base_kid, num_fake=1000, metric_batch_size=64):
    fid = copy.deepcopy(base_fid)
    kid = copy.deepcopy(base_kid)
    generated = 0
    pbar = tqdm(total=num_fake, desc='fake metric pass', leave=False)
    while generated < num_fake:
        batch_n = min(metric_batch_size, num_fake - generated)
        fake_images = prepare_for_inception_metrics(sample_fn(batch_n))
        fid.update(fake_images, real=False)
        kid.update(fake_images, real=False)
        generated += batch_n
        pbar.update(batch_n)
    pbar.close()
    kid_mean, kid_std = kid.compute()
    return {'fid': fid.compute().item(), 'kid_mean': kid_mean.item(), 'kid_std': kid_std.item()}

base_fid, base_kid = build_cached_real_metrics(train_loader, device)
wgan_scores = evaluate_with_cached_real(lambda n: 0.5 * (wgan_G(torch.randn(n, latent_dim, device=device)) + 1.0), base_fid, base_kid)
sn_scores = evaluate_with_cached_real(lambda n: 0.5 * (sn_G(torch.randn(n, latent_dim, device=device)) + 1.0), base_fid, base_kid)
cgan_scores = evaluate_with_cached_real(
    lambda n: 0.5 * (cG(torch.randn(n, latent_dim, device=device), torch.randint(0, num_classes, (n,), device=device)) + 1.0),
    base_fid, base_kid,
)
print({'wgan_gp': wgan_scores, 'sn_gan': sn_scores, 'cgan': cgan_scores})


After seeing the code, the final perspective should be clearer. `WGAN-GP` is about critic geometry. `Spectral normalization` is about critic control. `Conditional GANs` are about changing the generation task itself. The variants are not random additions to the literature. They are targeted responses to distinct weaknesses of the original adversarial formulation.

## A Brief Outlook Beyond the Core Set

There are, of course, many other GAN variants: **CycleGAN** for unpaired translation, **StyleGAN** for highly structured latent control, and large class-conditional systems such as **BigGAN**. These matter historically and practically, but for a first three-hour GAN block it is usually better to treat them as extensions of the three central ideas already discussed.

- CycleGAN extends the conditional viewpoint to unpaired domain translation.
- StyleGAN extends the architectural-control viewpoint to multi-scale latent modulation.
- BigGAN extends the conditional and stabilization story to large-scale class-conditioned generation.

Once students understand WGAN-GP, spectral normalization, and cGANs, they are in a good position to read these larger models without getting lost.